# Python Warm-Up: Foundations for Acrobot (pygame + Gymnasium)

Before we touch pygame or Gymnasium, get comfortable with the handful of Python patterns that show up *everywhere* in both projects. This is not a general Python tour — every section here maps directly to something you'll see in the "real" content next session.

**The one idea everything today builds toward:**

```
choose an action  ->  send it to the world  ->  get a result back  ->  update state  ->  repeat
```

This loop is the heartbeat of a pygame game loop *and* a Gymnasium RL loop. Keep it in your head all session.


---
## Section 1: Loops, State, and "Feeding Output Back In"

The biggest conceptual trip-up isn't loop *syntax* — it's the idea of a loop where **each iteration's output becomes the next iteration's input.** That's what makes something feel like a simulation instead of just "doing a thing 10 times."

### `while True` + `break` vs. `for`

Use `for` when you know the number of repetitions ahead of time (e.g. "try 20 evaluation episodes").
Use `while True:` + `break` when you don't know how long something will run and instead want to stop the moment a *condition* becomes true (e.g. "run until the episode ends").

```python
while True:
    # do something
    if some_condition:
        break
```

### State feeding into itself

```python
state = start
while True:
    state = update(state)   # this iteration's output becomes next iteration's input
    if done(state):
        break
```

This is exactly the shape of `state = next_state` you'll see later in the Q-learning loop, and of updating a ball's position/velocity each frame in pygame.


In [1]:
position = 0
velocity = 2
steps = 0

while True:
    position += velocity   # <-- output feeds back in as next input
    steps += 1

    if position > 100:
        break

print(f"Crossed 100 after {steps} steps. Final position: {position}")


jo


### 🧪 Exercise 1

Write a loop that starts a ball at `position = 0` with `velocity = 3`.

- Each tick, update `position`.
- Every 5th step, increase `velocity` by 1 (the ball speeds up — like gravity, sort of).
- Stop the loop once `position > 50`, and print how many steps it took.

Use a `while True:` + `break` — no `for` loop here, since we don't know in advance how many steps it'll take.


In [1]:
position = 0
velocity = 3
steps = 0

while True:
    position += velocity   # <-- output feeds back in as next input
    steps += 1

    if steps % 5 == 0:
        velocity += 1

    if position > 50:
        break

print(f"Crossed 50 after {steps} steps. Final position: {position}")


Crossed 50 after 14 steps. Final position: 55


---
## Section 2: Functions, Defaults, and Multi-Value Returns

Two patterns you'll see constantly:

**1. Default arguments** — lets you call a function with or without overriding certain values:
```python
def step(state, velocity=1):
    ...
```

**2. Returning multiple values, then unpacking them** — a function can return several things at once as a tuple, and you can unpack them directly into named variables:
```python
def move(x, y):
    return x + 1, y + 2

new_x, new_y = move(3, 4)   # tuple unpacking — same trick as `a, b = 1, 2`
```

You'll see this exact shape in Gymnasium: `observation, reward, terminated, truncated, info = env.step(action)` — one function call, five values back, all unpacked in one line.

### Combining stop conditions with `or`

```python
done = terminated or truncated   # True if EITHER is True
```


### Live demo

Run this together. `move()` shows both patterns from above at once: it has a **default argument** (`step=1`) and it **returns two values** that we unpack in the loop.

Watch how the loop looks almost identical to Section 1's ball loop — the only difference is that the update logic now lives inside a function instead of being written inline.

In [ ]:
def move(x, step=1):
    new_x = x + step
    crossed_zero = new_x > 0 and x <= 0  # True the instant we cross from <= 0 to >0
    return new_x, crossed_zero


x = -3
for _ in range(5):
    x, crossed = move(x)  # tuple unpacking
    print(f"x={x}, crossed_zero={crossed}")

# calling with the default vs. overriding it:
x2, _ = move(10)  # uses default step=1
x3, _ = move(10, step=5)  # overrides the default
print("x2 (default step):", x2)
print("x3 (step=5):", x3)

### 🧪 Exercise 2

Now extend the live demo above: instead of `move(x, step=1)`, refactor the **ball simulation** from Section 1 into a function the same way.

Write `def step(position, velocity):` that:
- returns the **new position**, and
- a boolean `done` that's `True` once `position > 50`

(just like `move()` returned `new_x` and `crossed_zero` together)

Then write a loop that calls `step(...)`, unpacks the result — same pattern as `x, crossed = move(x)` — updates `position`, and breaks when `done` is `True`.

In [ ]:
def step(position, velocity):
    new_position = position + velocity
    done = new_position > 50
    return new_position, done

position = 0
velocity = 3
steps = 0

while True:
    position, done = step(position, velocity)
    steps += 1
    if done:
        break

print(f"Done after {steps} steps. Final position: {position}")

---
## Section 3: Dictionaries as Lookup Tables, Tuples as Keys

This is the pattern most people haven't used much before today, and it's central to how a Q-table works later — so it's worth slowing down for.

### The "look up, or create if missing" pattern

```python
Q = {}                     # empty dictionary: nothing known yet

if state not in Q:
    Q[state] = 0            # create an entry the first time we see this state

Q[state] += 1               # now safe to update it
```

This lets you store information **only for states you've actually seen**, instead of pre-allocating a giant table for every possible state up front (which is often impossible — imagine trying to list every possible sensor reading!).

### Why tuples (not lists) as dictionary keys

Dictionary keys must be **hashable** — essentially, "unchangeable" (immutable). Tuples are immutable (`(2, 1, 4)`), so they work as keys. Lists are mutable (`[2, 1, 4]` can change after creation), so Python won't allow them as keys at all — you'll get an error if you try.

```python
position = (2, 1, 4)        # tuple: OK as a dict key
# position = [2, 1, 4]      # list: NOT allowed as a dict key
```


In [ ]:
import random

visit_counts = {}   # maps (x, y) -> number of visits

for _ in range(20):
    coord = (random.randint(0, 2), random.randint(0, 2))   # a random (x, y) tuple

    if coord not in visit_counts:  
        visit_counts[coord] = 0

    visit_counts[coord] += 1

for coord, count in visit_counts.items():
    print(f"{coord}: visited {count} time(s)")


(1, 1): visited 3 time(s)
(2, 1): visited 4 time(s)
(0, 1): visited 6 time(s)
(1, 2): visited 2 time(s)
(0, 0): visited 3 time(s)
(1, 0): visited 1 time(s)
(2, 0): visited 1 time(s)


### 🧪 Exercise 3

Write a function `get_or_create(table, key)` that:
- returns `table[key]` if `key` is already in `table`
- otherwise, creates `table[key] = []` (an empty list), stores it, and returns it

Then use it to build a dictionary that groups a list of `(x, y)` points by which "bucket" they fall in — where the bucket is `(x // 10, y // 10)` (integer division, like a coarse grid). Append each point to the list for its bucket.

This is structurally identical to `get_q_values()` in the Acrobot notebook — "look up, or create a fresh default, then use it."


In [ ]:
points = [(3, 4), (12, 7), (5, 5), (13, 19), (25, 3), (4, 8), (11, 11)]

def get_or_create(table, key):
    if key not in table:
        table[key] = []
    return table[key]

buckets = {}

for x, y in points:
    bucket_key = (x // 10, y // 10) 
    bucket_list = get_or_create(buckets, bucket_key)
    bucket_list.append((x, y))

print(buckets)

---
## Section 4: NumPy Essentials

Only the handful of operations you'll actually see later:

| Function | What it does | Where you'll see it |
|---|---|---|
| `np.zeros(n)` | Array of `n` zeros | Initializing Q-values for a new state |
| `np.argmax(array)` | Index of the largest value | "Which action has the best Q-value?" |
| `np.linspace(low, high, n)` | `n` evenly spaced numbers between `low` and `high` | Defining bin edges for discretization |
| `np.digitize(value, edges)` | Which bin a value falls into, given edges | Turning a continuous reading into a bin index |


In [9]:
import numpy as np

# np.zeros -- a fresh row of "unknown" values, one per action
q_values = np.zeros(3)
print("q_values:", q_values)

# np.argmax -- which index holds the largest value?
q_values = np.array([1.2, 5.7, 3.3])
best_action = np.argmax(q_values)
print("best_action:", best_action)


q_values: [0. 0. 0.]
best_action: 1
